# Prompt Evaluation Workflow

1. **Draft a Prompt**
   - Write the initial prompt and define the expected output.

2. **Create an Evaluation Dataset**
   - Prepare test cases with sample inputs and expected outputs.

3. **Feed Through Claude**
   - Run the prompt through Claude using the evaluation dataset.
   - Collect and record the generated outputs.

4. **Feed Through Grader**
   - Evaluate Claude's outputs against predefined criteria.
   - Measure accuracy, relevance, consistency, and instruction adherence.

5. **Change Prompt and Repeat**
   - Analyze evaluation results and identify weaknesses.
   - Refine the prompt based on the grader's feedback.
   - Repeat the evaluation workflow until the desired performance is achieved.

In [2]:
# load env variables from .env file
from dotenv import load_dotenv
load_dotenv()
import os

In [3]:
# create api client 
from anthropic import Anthropic
client = Anthropic(
    base_url = os.getenv("ANTHROPIC_BASE_URL"),
    api_key = os.getenv("ANTHROPIC_AUTH_TOKEN"),
)

model_name = os.getenv("ANTHROPIC_MODEL")

In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model_name,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [5]:
import json

import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for prompts that generate Python, JSON, or Regex
for AWS-related tasks.

Requirements:
- Return a JSON array containing exactly 3 objects.
- Each object must contain a "task" field describing an AWS-related task.
- Each task must be solvable using a single Python function, a single JSON object,
  or a single regex.
- Keep tasks short and simple.
- Return only valid JSON without Markdown code fences.

Example:
[
    {"task": "Write a Python function to validate an S3 bucket name."},
    {"task": "Create a JSON object representing an AWS Lambda configuration."},
    {"task": "Write a regex to match an AWS ARN."}
]
"""

    messages = []
    add_user_message(messages, prompt)

    text = chat(messages)

    dataset = json.loads(text)

    if len(dataset) != 3:
        raise ValueError(f"Expected 3 tasks, but received {len(dataset)}")

    return dataset

In [6]:
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [7]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [8]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [9]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results

In [10]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [11]:
print(json.dumps(results, indent=2))

[
  {
    "output": "To create a Python function that checks the validity of an S3 bucket name, we need to adhere to the naming conventions for S3 buckets, which include some specific rules:\n\n- The bucket name must be 3-63 characters long.\n- It can only contain lowercase letters, numbers, and hyphens.\n- The first and last characters must be alphanumeric (letters or numbers).\n\nHere is a Python function that validates an S3 bucket name based on these rules:\n\n```python\nimport re\n\ndef is_valid_s3_bucket_name(bucket_name):\n    \"\"\"\n    Check if the given bucket name is valid for an S3 bucket.\n    \n    Args:\n    bucket_name (str): The name of the S3 bucket to validate.\n    \n    Returns:\n    bool: True if the bucket name is valid, False otherwise.\n    \"\"\"\n    # Regular expression to match valid bucket names\n    valid_pattern = re.compile(r\"^[a-z0-9]([-a-z0-9]{0,61}[a-z0-9])$\")\n\n    # Check if the bucket name matches the pattern\n    if valid_pattern.match(bucket